# HW3 練功本 v3：看懂 `st_minecode_v3.ipynb` 的 ensemble

這本是**給我自己看的**，不用交。v3 的檢索和 v2 一樣，只多了「3 個分類器一起判斷」，所以前 8 節只講 v3 **新增**的程式；
第 9、10 節是 v3 跑完之後，拿 dev 的標準答案回頭分析的結果。全部大約 25 分鐘做得完。
和前兩本一樣：只用幾行假資料，CPU 幾秒就跑完；每一節都是 **白話解說 → 🧪 示範 → ✍️ 手打練習 → 參考答案**。

為什麼 v3 也要一本？v3 的新程式（`ensemble_scores`、模型組合、`pred_*.json`）在正式 notebook 裡，
要先把 3 個模型訓練完（1.5～2 小時）才跑得到，沒辦法「改一行、跑一下」。這裡用假的 logits 代替，馬上就能看到結果。

練習格子跑的時候會自動對答案：⏳ = 還沒寫、✅ = 對了、❌ = 不對（會印出期望的結果）。

**v3 的實際結果**（Colab L4 跑完的數字，後面幾節會用到）：

| | v1 | v2 | v3 |
| :--- | :---: | :---: | :---: |
| dev Evidence Recall@5 | 82.70% | 85.85% | 85.50% |
| dev Accuracy | 67.83% | 73.30% | 74.77% |
| dev FEVER score | 62.27% | 69.00% | **70.03%** |
| Kaggle public | 0.776 | 0.856 | **0.857** |

| 節 | 主題 | 對應 v3 notebook 的位置 | 建議 |
| :--- | :--- | :--- | :--- |
| 1 | ensemble 為什麼有用（含 v3 實測） | 第 4 節開頭說明 | 看懂就好 |
| 2 | 平均「機率」vs 平均 logits | 第 5 節開頭說明 | 看懂就好 |
| 3 | `ensemble_scores`：softmax → 平均 → log | 第 5 節 `ensemble_scores` | ✍️ **手打** |
| 4 | 3 個模型有幾種組合、怎麼選（含 v3 實測） | 第 5 節 `combinations` 迴圈 | ✍️ **手打** |
| 5 | 輸入指紋：什麼時候可以沿用存檔 | 第 4 節 `input_fingerprint` / `same_setting` | ✍️ **手打** |
| 6 | 學不起來的模型自動排除 | 第 4 節「依序訓練」最後 | 看懂就好 |
| 7 | 為什麼只凍結 xlm-roberta 的詞向量 | 第 1 節 `CLASSIFIER_MODELS`、`train_classifier` | 看懂就好 |
| 8 | public 分數的誤差有多大（v2 0.856 vs v3 0.857） | 決定要提交哪一版 | ✍️ **手打** |
| 9 | 在 dev 上調參會高估多少：交叉驗證 | v3 跑完後的分析（正式 notebook 沒有） | ✍️ **手打** |
| 10 | 分數還掉在哪裡：要好幾頁才能證明的題目 | v3 跑完後的分析（正式 notebook 沒有） | ✍️ **手打** |

In [ ]:
import hashlib
import math
import random
from itertools import combinations

import torch
import torch.nn.functional as F

torch.manual_seed(0)


def check(name, got, expected):
    """練習格子用的對答案小工具。"""
    if got is None or got is Ellipsis:
        print(f'⏳ {name}：還沒寫完（把 ... 換成你的程式）')
        return
    same = torch.allclose(got, expected, atol=1e-5) if isinstance(expected, torch.Tensor) else got == expected
    print(f'✅ {name}：正確！' if same else f'❌ {name}：不對\n   你的結果：{got}\n   期望結果：{expected}')

## 1. ensemble 為什麼有用？（看懂就好）

3 個模型各自有 80% 的題目答對。如果它們**錯在不同的題目**，把 3 個模型的意見平均，
某一題只有 1 個模型錯的時候，另外 2 個會把它拉回來，合起來就比任何一個都準。

但如果 3 個模型**錯在同樣的題目**，平均也救不回來。所以 v3 挑的 3 個模型刻意不一樣：

| 模型 | 和其他兩個哪裡不一樣 |
| :--- | :--- |
| chinese-roberta-wwm-ext-large | v2 用的那個 |
| chinese-macbert-large | 一樣大，但預訓練時遮字的方式不同（用相似詞代替 [MASK]） |
| xlm-roberta-large | 多語言、詞表 25 萬、斷詞方式完全不同 |

In [ ]:
# 🧪 示範：10 題，正確答案都是第 0 類；每個模型錯 2 題（錯的時候把機率給第 1 類）
def fake_probs(wrong_questions):
    rows = []
    for question in range(10):
        rows.append([0.3, 0.6, 0.1] if question in wrong_questions else [0.7, 0.2, 0.1])
    return torch.tensor(rows)


def accuracy(probs):
    return round((probs.argmax(dim=1) == 0).float().mean().item(), 2)


for title, wrongs in [('錯在不同題目', [{0, 1}, {2, 3}, {4, 5}]), ('錯在同樣題目', [{0, 1}, {0, 1}, {0, 1}]),
                      ('部分重疊', [{0, 1}, {1, 2}, {3, 4}])]:
    models = [fake_probs(w) for w in wrongs]
    average = torch.stack(models).mean(dim=0)
    print(f'{title}：單一模型 {[accuracy(m) for m in models]} → 平均後 {accuracy(average)}')

### v3 實際上是哪一種？

v3 跑完後，我拿 dev 的標準答案對 3 個模型各自的預測（`pred_*.json` 裡的 dev logits）：

| | 數字 |
| :--- | :---: |
| 單一模型的 dev Accuracy（roberta / macbert / xlmr） | 72.30% / 72.90% / 72.27% |
| 3 個模型平均後（還沒調 NEI bias、門檻） | 74.53% |
| 3 個模型**意見完全一樣**的題目 | 73.5% |
| 　└ 這些題目的準確率 | 82.4% |
| 3 個模型意見不一樣的題目的準確率 | 52.8% |
| 　└ 這些題目裡「**至少有 1 個模型答對**」的比例 | 84.8% |
| 全部題目裡「至少有 1 個模型答對」（ensemble 的理論上限） | 83.2% |

所以 v3 比較接近上面的「**部分重疊**」：3 個模型有四分之三的題目意見一樣，錯也常錯在同一題，平均只多了約 2 個百分點的 Accuracy。
上限 83.2% 和實際 74.53% 差很多，是因為意見不一樣的題目幾乎都是「2 個說 A、1 個說 B」，而答對的常常是少數的那 1 個，平均的時候就輸了。
想再往上，要找**錯的題目更不一樣**的模型（例如不同類型的架構），而不是再多加一個差不多的 BERT。

In [ ]:
# 🧪 示範：意見一樣 / 不一樣的題目各佔多少，對整體準確率的影響（數字是 v3 dev 的實測值）
agree_ratio, agree_acc, disagree_acc = 0.7347, 0.8240, 0.5276
overall = agree_ratio * agree_acc + (1 - agree_ratio) * disagree_acc
print(f'整體 = {agree_ratio:.1%} × {agree_acc:.1%} + {1 - agree_ratio:.1%} × {disagree_acc:.1%} ≈ {overall:.1%}')
print(f'意見一樣的題目已經有 {agree_acc:.0%} 準，ensemble 真正能幫忙的只有意見不一樣的那 {1 - agree_ratio:.1%} 題。')

## 2. 平均「機率」vs 平均 logits（看懂就好）

logits 的大小沒有上限，不同模型的「自信程度」也不一樣。如果某個模型特別愛講大數字，直接平均 logits，
它一個人就能蓋過另外兩個。softmax 之後每個模型的三類機率加起來都是 1，大家的「票」一樣重。

In [ ]:
# 🧪 示範：A 很有自信地說 SUPPORTS（logit 10），B、C 都說 REFUTES，但只有 logit 2
LABELS = ['SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO']
logits_a = torch.tensor([[10.0, 0.0, 0.0]])
logits_b = torch.tensor([[0.0, 2.0, 0.0]])
logits_c = torch.tensor([[0.0, 2.0, 0.0]])
mean_logits = torch.stack([logits_a, logits_b, logits_c]).mean(dim=0)
mean_probs = torch.stack([x.softmax(dim=1) for x in (logits_a, logits_b, logits_c)]).mean(dim=0)


def show(tensor, digits=3):
    return [round(x, digits) for x in tensor.flatten().tolist()]


print('平均 logits：', show(mean_logits, 2), '→', LABELS[mean_logits.argmax().item()])
print('平均機率：  ', show(mean_probs), '→', LABELS[mean_probs.argmax().item()])
print('A 一個人就決定了「平均 logits」的結果；平均機率時，2 票 REFUTES 贏 1 票 SUPPORTS。')

# 🧪 示範：logits 全部加同一個數，softmax 不變（所以 logits 的「絕對大小」本來就沒有意義）
print('softmax([1, 2, 3])  =', show(torch.tensor([1., 2., 3.]).softmax(dim=0), 4))
print('softmax([11,12,13]) =', show(torch.tensor([11., 12., 13.]).softmax(dim=0), 4))

## 3. `ensemble_scores`：softmax → 平均 → log ✍️

v3 的 `ensemble_scores` 只有 2 行，做 4 件事：

```text
每個模型的 logits [題數, 3] ──softmax──► 機率 [題數, 3]
       ──torch.stack──► [模型數, 題數, 3] ──mean(dim=0)──► [題數, 3]（沿「模型」那一維平均）
       ──clamp_min(1e-9)──► 把 0 換成很小的數 ──log──► 新的分數 [題數, 3]
```

- **為什麼最後要取 log？** 後面的 NEI bias 是「加在分數上」，和 v2 一樣。只有 1 個模型時，log(softmax(logits)) = logits − 常數，
  取 argmax 的結果和原本完全一樣，所以單一模型和 v2 的做法一致。
- **`clamp_min(1e-9)` 在防什麼？** fp16 算出來的機率可能剛好是 0，`log(0) = -inf`，後面加 bias、比大小都會出問題。

In [ ]:
# 🧪 示範：log(0) 的問題
print('log([0, 0.5])               =', show(torch.tensor([0.0, 0.5]).log()))
print('clamp_min(1e-9) 之後再 log =', show(torch.tensor([0.0, 0.5]).clamp_min(1e-9).log()))

In [ ]:
# ✍️ 手打練習：ensemble_scores(logits_list) → [題數, 3]
model_a = torch.tensor([[2.0, 0.0, 1.0], [0.0, 3.0, 0.0]])   # 2 題、3 類
model_b = torch.tensor([[0.5, 0.0, 1.5], [1.0, 1.0, 0.0]])


def ensemble_scores(logits_list):
    probabilities = ...   # 每個 logits 做 softmax(dim=1)，再 torch.stack 成 [模型數, 題數, 3]
    average = ...         # 沿著第 0 維（模型）取平均
    return None           # 改成 average.clamp_min(1e-9).log()


check('ensemble_scores', ensemble_scores([model_a, model_b]),
      torch.tensor([[-0.802444, -2.161628, -0.828669], [-1.453295, -0.406644, -2.299386]]))
single = ensemble_scores([model_b])
check('只有 1 個模型時 argmax 不變', None if single is None else single.argmax(dim=1).tolist(), model_b.argmax(dim=1).tolist())

<details><summary>參考答案（先自己寫再點開）</summary>

```python
def ensemble_scores(logits_list):
    probabilities = torch.stack([logits.softmax(dim=1) for logits in logits_list])
    average = probabilities.mean(dim=0)
    return average.clamp_min(1e-9).log()
```
第 1 題：A 說 SUPPORTS（2.0）、B 說 NEI（1.5），平均後 SUPPORTS（-0.80）只贏 NEI（-0.83）一點點，這種題目就很吃 NEI bias。
正式 notebook 的版本多了一步 `torch.tensor(result[f'{split}_logits'])`，因為 logits 是從 `pred_*.json` 讀回來的 list。
</details>

## 4. 3 個模型有幾種組合？怎麼選？ ✍️

ensemble 不一定是越多越好：如果某個模型很弱，把它加進來反而會拖累。所以 v3 把所有組合都在 dev 上試一次：
單一模型 3 種、兩兩一組 3 種、三個一起 1 種，共 2³ − 1 = **7 種**。每一種各自挑最好的 NEI bias 和門檻，最後選 dev FEVER 最高的。

`itertools.combinations(list, k)` 會列出「從 list 裡挑 k 個、不管順序」的所有組合。

**同分時選模型多的**：dev 分數一樣的話，平均越多個模型，對沒看過的 test 通常越穩（比較不會剛好只適合 dev）。

In [ ]:
# 🧪 示範：列出所有組合
names = ['roberta', 'macbert', 'xlmr']
for size in range(1, len(names) + 1):
    print(f'{size} 個一組：', list(combinations(names, size)))

In [ ]:
# ✍️ 手打練習：pick_best_combo(rows) → 回傳最好的那個組合（tuple）
# rows = [(組合 tuple, dev FEVER), ...]；FEVER 最高的贏；FEVER 一樣（四捨五入到小數 6 位）時，模型多的贏
def pick_best_combo(rows):
    best = ...    # 提示：max(rows, key=lambda row: (round(row[1], 6), len(row[0])))
    return None   # 改成：best 的組合（row[0]）


rows = [(('roberta',), 0.690), (('macbert',), 0.685), (('xlmr',), 0.640),
        (('roberta', 'macbert'), 0.702), (('roberta', 'xlmr'), 0.695), (('macbert', 'xlmr'), 0.690),
        (('roberta', 'macbert', 'xlmr'), 0.700)]
check('FEVER 最高的贏', pick_best_combo(rows), ('roberta', 'macbert'))
rows_tie = rows[:-1] + [(('roberta', 'macbert', 'xlmr'), 0.702)]
check('同分選模型多的', pick_best_combo(rows_tie), ('roberta', 'macbert', 'xlmr'))

<details><summary>參考答案</summary>

```python
def pick_best_combo(rows):
    best = max(rows, key=lambda row: (round(row[1], 6), len(row[0])))
    return best[0]
```
`key` 回傳 tuple 時，Python 先比第 1 個元素，一樣才比第 2 個。`round(..., 6)` 是為了避免 0.7020000001 和 0.702 這種浮點誤差被當成不一樣。
這個例子裡 xlmr 單獨最差，但「roberta + xlmr」（0.695）還是比 roberta 單獨（0.690）好：弱的模型只要錯的題目和別人不一樣，還是有幫助。
</details>

In [ ]:
# 🧪 v3 實測：第 5 節印出來的 7 種組合（dev FEVER，各自調過 NEI bias 和門檻）
v3_rows = [(('roberta',), 0.6807), (('macbert',), 0.6900), (('xlmr',), 0.6787),
           (('roberta', 'macbert'), 0.6950), (('roberta', 'xlmr'), 0.6873), (('macbert', 'xlmr'), 0.6957),
           (('roberta', 'macbert', 'xlmr'), 0.7003)]
for combo, score in sorted(v3_rows, key=lambda row: -row[1]):
    print(f'{" + ".join(combo):26s} {score:.2%}')
best = max(v3_rows, key=lambda row: (round(row[1], 6), len(row[0])))   # 和 pick_best_combo 同一個規則
print('選用：', ' + '.join(best[0]))
print('xlmr 單獨最差（67.87%），但 macbert + xlmr（69.57%）比 roberta + macbert（69.50%）還好：錯的題目不一樣，就有幫助。')

## 5. 輸入指紋：什麼時候可以沿用存檔？ ✍️

3 個模型要訓練 1.5 小時以上，Colab 很可能中途斷線。所以 v3 每訓練完一個模型，就把 dev / test 的 logits 存成 `pred_模型名.json`，
重跑時直接讀檔。但不能「有檔案就讀」：如果中間重跑過檢索（證據變了），或改了學習率，舊的 logits 就是錯的。

所以檔案裡多存一個**輸入指紋**：把 dev / test 的 claim 和證據文字接起來，算 MD5 取前 12 碼。內容只要差一個字，指紋就完全不同。
讀檔前檢查「指紋、epoch 數、學習率、要不要凍結詞向量」4 項都一樣，才沿用；有任何一項不一樣就重訓。

In [ ]:
# 🧪 示範：差一個字，指紋就完全不同
def fingerprint(texts):
    return hashlib.md5('\n'.join(texts).encode('utf-8')).hexdigest()[:12]


print(fingerprint(['周杰倫在2003年發行葉惠美', '【葉惠美】葉惠美是周杰倫的專輯']))
print(fingerprint(['周杰倫在2003年發行葉惠美', '【葉惠美】葉惠美是周杰倫的專輯。']), '← 只多了一個句號')

In [ ]:
# ✍️ 手打練習：same_setting(saved, spec, current_fingerprint, epochs) → True / False
# saved：從 pred_*.json 讀回來的 dict；spec：CLASSIFIER_MODELS 裡的一個模型設定
# 4 項都一樣才回傳 True：fingerprint、epochs、lr、freeze_embeddings（saved 裡可能缺欄位，用 .get）
def same_setting(saved, spec, current_fingerprint, epochs):
    return None   # 改成 4 個條件用 and 接起來


spec = {'name': 'FacebookAI/xlm-roberta-large', 'lr': 8e-6, 'freeze_embeddings': True}
saved = {'fingerprint': 'a1b2c3d4e5f6', 'epochs': 2, 'lr': 8e-6, 'freeze_embeddings': True}
check('全部一樣 → 沿用', same_setting(saved, spec, 'a1b2c3d4e5f6', 2), True)
check('檢索結果變了 → 重訓', same_setting(saved, spec, 'ffffffffffff', 2), False)
check('改了學習率 → 重訓', same_setting(saved, dict(spec, lr=5e-6), 'a1b2c3d4e5f6', 2), False)
check('舊檔沒有 epochs 欄位 → 重訓', same_setting({'fingerprint': 'a1b2c3d4e5f6', 'lr': 8e-6, 'freeze_embeddings': True},
                                                spec, 'a1b2c3d4e5f6', 2), False)

<details><summary>參考答案</summary>

```python
def same_setting(saved, spec, current_fingerprint, epochs):
    return (saved.get('fingerprint') == current_fingerprint and saved.get('epochs') == epochs
            and saved.get('lr') == spec['lr'] and saved.get('freeze_embeddings') == spec['freeze_embeddings'])
```
用 `saved.get('epochs')` 而不是 `saved['epochs']`：舊檔缺這個欄位時 `.get` 回傳 `None`，比較結果是 False → 重訓；`saved['epochs']` 會直接 `KeyError` 當掉。
正式 notebook 的 `same_setting(saved, spec)` 少了後 2 個參數，因為它直接讀全域變數 `fingerprint` 和 `EPOCHS`。
</details>

## 6. 學不起來的模型自動排除（看懂就好）

large 模型偶爾會「學不起來」：loss 不降，最後每一題都猜同一類。這種模型放進 ensemble 只會拖累，所以 v3 訓練完會檢查：
**每個 epoch 的 dev Accuracy 最高都不到 45%**，就標記成 `collapsed`，不放進 ensemble。

為什麼是 45%？三類大約各佔三分之一，全部猜同一類大概 33～40%；真的有學起來的模型（v2 是 73%）遠高於這個數。
第 4 節的組合比較也會自動避開它（它根本不在 `usable_results` 裡）。想救的話，把它的 `lr` 調小（例如 5e-6）重跑，設定變了會自動重訓（第 5 節）。

In [ ]:
# 🧪 示範
results = [{'name': 'roberta', 'accs': [0.71, 0.73]}, {'name': 'macbert', 'accs': [0.70, 0.72]},
           {'name': 'xlmr', 'accs': [0.38, 0.39]}]
for result in results:
    result['collapsed'] = max(result['accs']) < 0.45
usable = [result['name'] for result in results if not result['collapsed']]
print('可以用的模型：', usable)

## 7. 為什麼只凍結 xlm-roberta 的詞向量？（看懂就好）

凍結詞向量是為了省 GPU 記憶體（v2 練功本 §5）。省多少，要看詞表多大：

In [ ]:
for name, vocab, total in [('chinese-roberta-wwm-ext-large', 21128, 326e6), ('chinese-macbert-large', 21128, 326e6),
                           ('xlm-roberta-large', 250002, 560e6)]:
    embedding = vocab * 1024
    print(f'{name:30s} 詞表 {vocab:>7,}：詞向量 {embedding / 1e6:5.0f}M，佔全部 {embedding / total:5.1%}，'
          f'凍結可省 {embedding * 12 / 1024 ** 3:.1f} GB（梯度 + Adam 狀態）')

兩個中文模型的詞向量只佔 7%，凍結省不了多少，還可能少學一點；xlm-roberta 的詞向量佔 46%，凍結後才放得進 GPU。

另一個細節：`train_classifier` 把 `trainable`（只有 `requires_grad=True` 的參數）**同時**交給 AdamW 和 `clip_grad_norm_`。
被凍結的參數沒有梯度（`.grad` 是 `None`），本來就不用更新、也不用裁剪，只給 `trainable` 最清楚。

In [ ]:
# 🧪 示範：凍結的參數 .grad 是 None
layer = torch.nn.Linear(2, 1)
layer.bias.requires_grad = False
layer(torch.ones(1, 2)).sum().backward()
print('weight.grad =', layer.weight.grad.tolist(), '| bias.grad =', layer.bias.grad)

## 8. public 分數的誤差有多大？ ✍️

Kaggle public leaderboard 只用 test 的一小部分（v3 的說明寫約 17%，大約 1,000 題）。每一題不是 0 分就是 1 分，
所以 public score 就像「抽 1,000 題來考」，運氣好壞會讓分數上下跳。統計上，答對率 p、題數 n 時，分數的**標準誤差**是

$$\text{SE} = \sqrt{\frac{p(1-p)}{n}}$$

大約有 95% 的機率，分數落在真實值 ± 2 個 SE 之內。v2 vs v3 的 public 差距如果小於這個範圍，就分不出誰比較好，要看 dev（3,000 題）。

**實際發生的事**：v2 是 0.856，v3 是 0.857，只差 0.001，也就是 1,000 題裡大約多對 1 題。這一節算完你就知道，這個差距遠小於 public 的誤差。

In [ ]:
# ✍️ 手打練習：standard_error(p, n)
def standard_error(p, n):
    return None   # 改成 math.sqrt(...)


result = standard_error(0.85, 1000)
check('public（約 1,000 題）', result if result is None else round(result, 4), 0.0113)
result = standard_error(0.69, 3000)
check('dev（3,000 題）', result if result is None else round(result, 4), 0.0084)

<details><summary>參考答案</summary>

```python
def standard_error(p, n):
    return math.sqrt(p * (1 - p) / n)
```
public 的 SE 約 1.1 分，± 2 SE 就是 ± 2.3 分。v3 的 0.857 比 v2 的 0.856 只高 0.1 分，完全在誤差裡面，public 分不出兩版誰好。
dev 題數是 public 的 3 倍，SE 比較小，而且 v2、v3 考的是**同一批 3,000 題**（不是各抽各的），所以 dev 上 +1.03 分比 public 的 +0.1 分可靠。
但要注意：v3 是完整重跑，reranker 重新微調過，證據 Recall 從 85.85% 變成 85.50%，所以兩版的證據不是完全一樣，+1.03 分裡也混了一點檢索的差異。
</details>

In [ ]:
# 🧪 v3 實測：public 差 0.001 到底是幾題？
public_n = 1000
print(f'0.857 − 0.856 = 0.001 → 大約 {0.001 * public_n:.0f} 題')
se = math.sqrt(0.857 * (1 - 0.857) / public_n)
print(f'public 的 ±2 SE = ±{2 * se:.3f}，也就是 ±{2 * se * public_n:.0f} 題：差 1 題完全看不出來')
print('dev：v2 69.00% → v3 70.03%，3,000 題裡多對 31 題')

In [ ]:
# 🧪 示範：同一個「真實答對率 85%」的模型，抽不同的 1,000 題當 public，分數會差多少？
rng = random.Random(0)
answers = [1] * 5100 + [0] * 900          # 6,000 題 test，真實答對 85%
scores = []
for _ in range(10):
    public = rng.sample(answers, 1000)
    scores.append(sum(public) / 1000)
print('10 次抽樣的 public score：', [f'{s:.3f}' for s in scores])
print(f'最高 {max(scores):.3f}、最低 {min(scores):.3f}：模型完全一樣，分數就能差這麼多')

## 9. 在 dev 上調參，分數會高估多少？交叉驗證 ✍️

v3 的 NEI bias、門檻、模型組合、epoch 全都是在 dev 上挑的，報告的也是**同一份 dev** 的分數。
這就像「先看過考卷再決定答題策略」，報出來的分數會比真實情況樂觀一點（報告第 4 題的「限制」也寫了這件事）。

要知道到底樂觀多少，可以用 **k 折交叉驗證**（k-fold cross validation）：把 dev 切成 k 份，
每次用其中 k − 1 份挑參數，在剩下那 1 份（沒參與挑選的）上算分數，輪 k 次取平均。每一題都剛好當過一次「沒看過的考題」。

In [ ]:
# 🧪 示範：一個「根本沒用」的門檻規則——分數和答案完全無關，不管門檻設多少，真實準確率都是 50%
# 隨機產生 100 份「dev」（每份 300 題），每份都用兩種方式估分數，最後取平均
rng = random.Random(0)
thresholds = [i / 25 for i in range(26)]


def accuracy_at(rows, threshold):
    return sum((score >= threshold) == label for score, label in rows) / len(rows)


def best_threshold(rows):
    return max(thresholds, key=lambda threshold: accuracy_at(rows, threshold))


in_sample, cross_validated = [], []
k = 5
for _ in range(100):
    data = [(rng.random(), rng.random() < 0.5) for _ in range(300)]   # (分數, 答案是不是 1)
    in_sample.append(accuracy_at(data, best_threshold(data)))
    fold_scores = []
    for i in range(k):
        train_rows = [row for j, row in enumerate(data) if j % k != i]
        test_rows = [row for j, row in enumerate(data) if j % k == i]
        fold_scores.append(accuracy_at(test_rows, best_threshold(train_rows)))
    cross_validated.append(sum(fold_scores) / k)
print(f'在全部 300 題上挑門檻、再在同樣 300 題上算分：平均 {sum(in_sample) / 100:.1%}  ← 明明真實只有 50%')
print(f'5 折交叉驗證：                                平均 {sum(cross_validated) / 100:.1%}  ← 接近真實的 50%')

In [ ]:
# ✍️ 手打練習：kfold_indices(n, k) → [(train 的 index list, test 的 index list), ...]，共 k 組
# 第 i 組的 test 是 0..n-1 裡「除以 k 餘 i」的那些，其他的都是 train
def kfold_indices(n, k):
    return None   # 提示：list comprehension，j % k == i 放 test、j % k != i 放 train


check('n=6, k=3', kfold_indices(6, 3), [([1, 2, 4, 5], [0, 3]), ([0, 2, 3, 5], [1, 4]), ([0, 1, 3, 4], [2, 5])])
folds = kfold_indices(10, 5)
check('每一題剛好當一次 test', None if folds is None else sorted(j for _, test in folds for j in test), list(range(10)))

<details><summary>參考答案</summary>

```python
def kfold_indices(n, k):
    return [([j for j in range(n) if j % k != i], [j for j in range(n) if j % k == i]) for i in range(k)]
```
實際分析時我用的是 sklearn 的 `RepeatedStratifiedKFold`：先打亂、每一折的三類比例都一樣，再重複 5 次取平均。概念和這裡完全一樣。
</details>

### v3 實測：「在全部 dev 上調參」vs 交叉驗證

v3 跑完後，我用 `pred_*.json` 裡的 dev logits 做了 5 折 × 5 次的交叉驗證（不用 GPU，幾分鐘跑完）：

| 做法 | 在全部 dev 上調參再算分 | 5 折 × 5 次交叉驗證 |
| :--- | :---: | :---: |
| v3 的規則（NEI bias + reranker 門檻） | 70.03% | 69.90% |
| logistic regression stacker（3 個模型的機率 + reranker 分數 + 證據特徵） | — | 70.13% |
| GBDT stacker（同樣的特徵） | — | 69.02% |

從這張表可以學到三件事：

1. **v3 的規則只高估了約 0.1 分**：它只從小小的格子裡挑 2 個數字，能「背」dev 的空間很小，所以報告裡的 70.03% 大致可信。
2. **stacker 沒有比較好**：logistic regression 只多 0.23 分，25 折裡只有 64% 的折比較好，這是雜訊等級的差距，所以 v3 沒有改用它。
3. **參數越多越危險**：GBDT 能學很複雜的規則，結果在沒看過的那一折反而**掉了將近 1 分**。在同一份 dev 上調越多參數，在全部 dev 上算出來的分數就越不能信。

## 10. 分數還掉在哪裡？要好幾頁才能證明的題目 ✍️

v2 和 v3 的漏斗（dev 3,000 題）：

| 掉在哪一步 | v2 | v3 |
| :--- | ---: | ---: |
| ① 文件：前 5 頁沒找齊證據頁 | 223 | 223 |
| ② 證據：頁面對了，前 5 句沒找齊 | 60 | 67 |
| ③ 分類：證據找齊了，標籤判錯 | 417 | **362** |
| NOT ENOUGH INFO 判錯 | 230 | 247 |

ensemble 讓第 ③ 步少錯 55 題，但第 ①② 步一共 290 題「證據沒找齊」，完全沒有改善。
我拿 dev 的標準證據回頭看這 290 題，發現關鍵在**一組標準證據需要幾頁、幾句**。

複習一下標準證據的格式（v1 練功本講過）：一題可以有好幾組，每組是一個 set `{(頁名, 句號), ...}`，找齊**任一組**就算命中。
所以一題有多難，要看「**最容易的那一組**」：頁數最少的那組，頁數一樣時再看句數最少的。

In [ ]:
# ✍️ 手打練習：easiest_group(groups) → (頁數, 句數)，最容易的那一組要幾頁、幾句
# groups = [{(頁名, 句號), ...}, ...]；先比頁數，頁數一樣再比句數
def easiest_group(groups):
    return None   # 提示：min((頁數, 句數) for group in groups)，頁數 = len({page for page, _ in group})


check('1 頁 1 句就夠', easiest_group([{('芝加哥', 0)}, {('芝加哥', 3), ('美國', 5)}]), (1, 1))
check('同一頁的 2 句', easiest_group([{('一貫道', 2), ('一貫道', 5)}, {('一貫道', 2), ('路中一', 0)}]), (1, 2))
check('一定要 2 頁', easiest_group([{('全羅北道', 0), ('全州市', 4)}]), (2, 2))

<details><summary>參考答案</summary>

```python
def easiest_group(groups):
    return min((len({page for page, _ in group}), len(group)) for group in groups)
```
tuple 比大小時先比第 1 個元素（頁數），一樣才比第 2 個（句數），所以 `min` 一次就能做到「先比頁數、再比句數」。
</details>

### v3 實測：句數越多，越難找齊

| 最容易的那一組要幾句 | dev 題數（SUPPORTS / REFUTES） | v3 證據找齊的比例 |
| :--- | ---: | ---: |
| 1 句 | 1,273 | 94.8% |
| 2 句 | 513 | 76.8% |
| 3 句以上 | 214 | 50.9% |

證據沒找齊的 290 題裡，有 **179 題**的每一組都要 **2 頁以上**。多頁特別難，因為：

- reranker 是**一句一句**和 claim 比，第二頁那句單獨看常常不像在講這個 claim，容易排不進前 5 句。
- 更常見的是**第二頁根本沒進前 5 頁**。

那些沒找到的頁跑去哪了？「頁面都在前 5 頁、只差句子沒排進前 5」的有 67 題；其他 223 題漏掉的頁（一題可能漏不只一頁）分成三種：

| 漏掉的頁名出現在… | 頁數 | 可以怎麼補 |
| :--- | ---: | :--- |
| **第一輪已經找到的證據句裡** | 102 | 拿證據句再做一次標題比對（第二輪檢索） |
| claim 裡（標題比對有找到，只是排不進前 5 頁） | 50 | 多看幾頁，例如前 10 頁 |
| 哪裡都沒出現（別名、口語，例如 claim 寫「爬蟲類」，頁名是「爬行動物」） | 98 | 只能靠向量檢索更準 |

In [ ]:
# 🧪 示範：第二輪檢索（示意）——拿第一輪找到的證據句，再做一次標題比對
titles = {'全羅北道', '全州市', '大韓民國', '首府', '美食'}
claim = '韓國行政道全羅北道的首府在聯合國教科文組織指定的美食城市之一賓州。'
first_hop_sentence = '全羅北道是大韓民國的一個道，首府爲全州市。'   # 示意用的句子


def match_titles(text, titles):
    return sorted({text[s:e] for s in range(len(text)) for e in range(s + 2, len(text) + 1) if text[s:e] in titles})


print('claim 裡的標題：      ', match_titles(claim, titles))
print('第一輪證據句裡的標題：', match_titles(first_hop_sentence, titles))
print('「全州市」只出現在證據句裡：claim 寫的是被改過的「賓州」，只看 claim 怎麼找都找不到這一頁。')

### 為什麼最後停在 v3？

我用 v3 的檢索結果在 CPU 上模擬了第二輪檢索（還沒用 GPU 重排，只看「標準證據頁有沒有進候選」）：

| 做法 | dev 頁面涵蓋率 |
| :--- | ---: |
| v3（前 5 頁） | 88.85% |
| ＋第二輪：從證據句裡的標題多補 10 頁 | 91.90% |
| ＋第二輪：從前 2 句證據句多補 20 頁 | 93.10% |

頁面涵蓋率多了 3～4 分，但這些頁還要被 reranker 排進前 5 句、分類器也要判對，估計 dev FEVER 只會多 1 分左右；
再加上最後用 train + dev 重訓，預估 public 大約 0.87～0.88。要到 0.90，dev 大約要 74%（比現在多 4 分），
目前找不到補得了這麼多的方法，而且還要多跑 3～4 小時 GPU，所以決定交 v3。

如果之後還想往上，最值得做的就是這一節的**第二輪檢索**；分類器那邊，要找「錯的題目和現在 3 個不一樣」的模型（第 1 節）。